# Final evaluation and error analysis

Run every trained adapter through the same lm-eval harness and
build the comparison table that goes in the README. Then generate
a small sample of completions on the held-out test split and
bucket them by failure mode for a qualitative read on what the
GRPO model is and isn't doing.

In [1]:
# Tell lm-eval which adapters to evaluate. None of these adapters
# are loaded yet — lm-eval will load each one in a separate subprocess.
!pip install -q -U bitsandbytes transformers trl peft accelerate datasets torchao
!pip install -q "lm-eval[math]"

# Mount Drive and restore everything
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
DRIVE = '/content/drive/MyDrive/llm_posttraining'

# Restore data
if not os.path.exists('/content/data/sft_test'):
    os.makedirs('/content/data', exist_ok=True)
    shutil.copytree(f'{DRIVE}/data/sft_train',    '/content/data/sft_train')
    shutil.copytree(f'{DRIVE}/data/sft_val',      '/content/data/sft_val')
    shutil.copytree(f'{DRIVE}/data/sft_test',     '/content/data/sft_test')
    shutil.copytree(f'{DRIVE}/data/grpo_prompts', '/content/data/grpo_prompts')

# Restore all checkpoints
os.makedirs('/content/checkpoints', exist_ok=True)

if not os.path.exists('/content/checkpoints/sft_final'):
    shutil.copytree(f'{DRIVE}/checkpoints/sft_final', '/content/checkpoints/sft_final')

if not os.path.exists('/content/checkpoints/dpo_final'):
    shutil.copytree(f'{DRIVE}/checkpoints/dpo_final', '/content/checkpoints/dpo_final')

if not os.path.exists('/content/checkpoints/grpo_C_final'):
    shutil.copytree(f'{DRIVE}/checkpoints/grpo_C_final', '/content/checkpoints/grpo_C_final')

# Login to HF
from huggingface_hub import login
login(token="hf_xxxxxxxxxxxxxxxxxxxx")  # paste your token

print('All restored. Ready for final evaluation.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 193.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 156.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 67.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 169.5 MB/s eta 0:00:00


In [2]:
import json, re, torch
from datasets import load_from_disk, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = 'Qwen/Qwen3-8B'
ADAPTERS = {
    'baseline': None,
    'sft':      '/content/checkpoints/sft_final',
    'dpo':      '/content/checkpoints/dpo_final',
    'grpo_A':   '/content/checkpoints/grpo_A_final',
    'grpo_B':   '/content/checkpoints/grpo_B_final',
    'grpo_C':   '/content/checkpoints/grpo_C_final',
}

In [4]:
import subprocess

def run_eval(model_args, output_tag, task, num_fewshot):
    cmd = [
        'lm_eval',
        '--model', 'hf',
        '--model_args', model_args,
        '--tasks', task,
        '--num_fewshot', str(num_fewshot),
        '--apply_chat_template',
        '--limit', '100',
        '--batch_size', '4',
        '--output_path', f'/content/results/{output_tag}_{task}',
    ]
    subprocess.run(cmd, check=True)

# Parse the lm-eval result JSONs and print the GSM8K scores.
# This is the comparison table for the README.
run_eval(f'pretrained={BASE_MODEL},dtype=bfloat16', 'final_baseline', 'gsm8k_cot', 8)

# SFT
run_eval(f'pretrained={BASE_MODEL},peft=/content/checkpoints/sft_final,dtype=bfloat16', 'final_sft', 'gsm8k_cot', 8)

# GRPO C
run_eval(f'pretrained={BASE_MODEL},peft=/content/checkpoints/grpo_C_final,dtype=bfloat16', 'final_grpo_C', 'gsm8k_cot', 8)

I am skipping Minerva, it is taking more than 3 hours on H100

In [10]:
import glob, json

def get_metric(result_dir, task, metric='exact_match,flexible-extract'):
    files = glob.glob(f'{result_dir}/**/*.json', recursive=True)
    if not files:
        print(f'No files found in {result_dir}')
        return None
    with open(files[0]) as f:
        data = json.load(f)
    return data.get('results', {}).get(task, {}).get(metric)

table = []
for tag in ['final_baseline', 'final_sft', 'final_grpo_C']:
    row = {
        'model': tag,
        'gsm8k': get_metric(f'/content/results/{tag}_gsm8k_cot', 'gsm8k_cot'),
    }
    table.append(row)
    print(row)

{'model': 'final_baseline', 'gsm8k': 0.16}
{'model': 'final_sft', 'gsm8k': 0.22}
{'model': 'final_grpo_C', 'gsm8k': 0.19}


In [11]:
# Generate one completion per test problem and bucket into:
#   correct      — \boxed{} matches gold
#   wrong_answer — \boxed{} present but wrong
#   no_boxed     — no \boxed{} in the response at all
#   reward_hack  — correct but suspiciously long (>1500 words)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map='auto', torch_dtype=torch.bfloat16
)
model = PeftModel.from_pretrained(base, '/content/checkpoints/grpo_C_final')
model.eval()

test_ds = load_from_disk('/content/data/sft_test').select(range(20))
SYSTEM = 'Please reason step by step, and put your final answer within \\boxed{}.'

def extract_boxed(text):
    m = re.search(r'\\boxed\{([^}]*)\}', text)
    return m.group(1).strip() if m else None

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [12]:
cases = {'correct': [], 'wrong_answer': [], 'no_boxed': [], 'reward_hack': []}

for ex in test_ds:
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user',   'content': ex['problem']}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=1024, do_sample=True, temperature=0.6, top_p=0.95)
    response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    pred = extract_boxed(response)
    gold = ex.get('final_answer', '')

    entry = {'problem': ex['problem'][:100], 'gold': gold, 'pred': pred, 'response_len': len(response.split()), 'response': response[:300]}

    if pred is None:
        cases['no_boxed'].append(entry)
    elif pred.strip().lower() == gold.strip().lower():
        cases['correct'].append(entry)
        # Print one example from each non-empty category. These are the
        # qualitative cases to discuss in the README error-analysis section.
        if len(response.split()) > 1500:
            cases['reward_hack'].append(entry)
    else:
        cases['wrong_answer'].append(entry)

for k, v in cases.items():
    print(f'{k}: {len(v)}')

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


correct: 1
wrong_answer: 0
no_boxed: 19
reward_hack: 0


In [13]:
# Save the bucketed cases for the README.
for category, examples in cases.items():
    if examples:
        print(f'\n=== {category.upper()} EXAMPLE ===')
        e = examples[0]
        print(f'Problem: {e["problem"]}')
        print(f'Gold:    {e["gold"]}')
        print(f'Pred:    {e["pred"]}')
        print(f'Length:  {e["response_len"]} words')
        print(f'Resp:    {e["response"][:200]}...')


=== CORRECT EXAMPLE ===
Problem: A3. The expression is $\frac{x^{n-1}}{x^{n}-2 x^{n-1}}-\frac{x^{n}}{x^{n+1}-4 x^{n-1}}$. Which expre
Gold:    B
Pred:    B
Length:  502 words
Resp:    <think>
Okay, so I have this algebra problem here, and I need to figure out which expression is equivalent to the given one. The expression is:

$\frac{x^{n-1}}{x^{n}-2 x^{n-1}} - \frac{x^{n}}{x^{n+1}...

=== NO_BOXED EXAMPLE ===
Problem: The lengths of the two legs of a right triangle are the two distinct roots of the quadratic $x^2 - 3
Gold:    34
Pred:    None
Length:  548 words
Resp:    <think>
Okay, so I need to find the hypotenuse of a right triangle where the legs are the roots of the quadratic equation x² - 36x + 70. Hmm, let's start by recalling some algebra. If the roots of a q...


In [14]:
# Save final analysis for README
with open('/content/results/error_analysis.json', 'w') as f:
    json.dump(cases, f, indent=2)
print('Error analysis saved.')

Error analysis saved.


In [16]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import shutil, os, json
DRIVE = '/content/drive/MyDrive/llm_posttraining'

# Save results
os.makedirs(f'{DRIVE}/results', exist_ok=True)
shutil.copytree('/content/results', f'{DRIVE}/results', dirs_exist_ok=True)

# Save error analysis
with open(f'{DRIVE}/results/error_analysis.json', 'w') as f:
    json.dump(cases, f, indent=2)

# Save final table
with open(f'{DRIVE}/results/final_table.json', 'w') as f:
    json.dump(table, f, indent=2)

print('All results and error analysis saved to Drive.')

Mounted at /content/drive
All results and error analysis saved to Drive.
